## Datensammlung: BFS-Hotellerie (Tabellen 201 und 101), Wetter, Geodaten.

In [5]:
import requests, json, pathlib

BASE = "https://www.pxweb.bfs.admin.ch/api/v1/de/px-x-1003020000_{t}/px-x-1003020000_{t}.px"
RAW = pathlib.Path("../data/raw/bfs"); RAW.mkdir(parents=True, exist_ok=True)

for t in ["201", "101"]:
    meta = requests.get(BASE.format(t=t), timeout=30).json()
    (RAW / f"meta_{t}.json").write_text(json.dumps(meta, ensure_ascii=False, indent=1), encoding="utf-8")
    print("Tabelle", t, "-", meta["title"])
    for v in meta["variables"]:
        print(f"  {v['code']:<14} {len(v['values']):>4} Werte   z.B. {v['valueTexts'][:4]}")

print()
print("--- Codes ---")
for t in ["201", "101"]:
    meta = json.loads((RAW / f"meta_{t}.json").read_text(encoding="utf-8"))
    print("Tabelle", t)
    for v in meta["variables"]:
        if v["code"] in ("Indikator", "Monat", "Herkunftsland"):
            print(" ", v["code"], list(zip(v["values"], v["valueTexts"]))[:15])



Tabelle 201 - Hotellerie: Angebot und Nachfrage der geöffneten Betriebe nach Jahr, Monat, Gemeinde und Indikator
  Jahr             14 Werte   z.B. ['2013', '2014', '2015', '2016']
  Monat            13 Werte   z.B. ['Jahrestotal', 'Januar', 'Februar', 'März']
  Gemeinde        186 Werte   z.B. ['Kloten', 'Opfikon', 'Dübendorf', 'Uster']
  Indikator         8 Werte   z.B. ['Betriebe', 'Zimmer', 'Betten', 'Ankünfte']
Tabelle 101 - Hotellerie: Ankünfte und Logiernächte der geöffneten Betriebe nach Jahr, Monat, Gemeinde, Herkunftsland und Indikator
  Jahr             14 Werte   z.B. ['2013', '2014', '2015', '2016']
  Monat            13 Werte   z.B. ['Jahrestotal', 'Januar', 'Februar', 'März']
  Gemeinde        186 Werte   z.B. ['Kloten', 'Opfikon', 'Dübendorf', 'Uster']
  Herkunftsland    73 Werte   z.B. ['Herkunftsland - Total', 'Schweiz', 'Deutschland', 'Frankreich']
  Indikator         2 Werte   z.B. ['Ankünfte', 'Logiernächte']

--- Codes ---
Tabelle 201
  Monat [('YYYY', 'Jahrestota

In [6]:
meta = json.loads((RAW / "meta_201.json").read_text(encoding="utf-8"))
ind = next(v for v in meta["variables"] if v["code"] == "Indikator")
for code, text in zip(ind["values"], ind["valueTexts"]):
    print(code, text)

1 Betriebe
2 Zimmer
3 Betten
4 Ankünfte
5 Logiernächte
6 Zimmernächte
7 Zimmerauslastung in %
8 Bettenauslastung in %


## 2. Hotellerie-Daten (Tabelle 201)

In [7]:
   import requests, json, time, pathlib
   import pandas as pd

   BASE = "https://www.pxweb.bfs.admin.ch/api/v1/de/px-x-1003020000_{t}/px-x-1003020000_{t}.px"
   RAW = pathlib.Path("../data/raw/bfs")
   meta = json.loads((RAW / "meta_201.json").read_text(encoding="utf-8"))
   gemeinden = next(v for v in meta["variables"] if v["code"] == "Gemeinde")["values"]

   def fetch_201(jahr):
       query = {
           "query": [
               {"code": "Jahr",      "selection": {"filter": "item", "values": [str(jahr)]}},
               {"code": "Monat",     "selection": {"filter": "item", "values": [str(m) for m in range(1, 13)]}},
               {"code": "Gemeinde",  "selection": {"filter": "item", "values": gemeinden}},
               {"code": "Indikator", "selection": {"filter": "item", "values": ["2","3","5","6","7","8"]}},
           ],
           "response": {"format": "json"},
       }
       r = requests.post(BASE.format(t="201"), json=query, timeout=60)
       r.raise_for_status()
       return r.json()

   for jahr in range(2013, 2027):
       ziel = RAW / f"hesta_201_{jahr}.json"
       if ziel.exists():
           continue                      # schon geladen -> überspringen
       data = fetch_201(jahr)
       ziel.write_text(json.dumps(data), encoding="utf-8")
       print(jahr, len(data["data"]), "Zeilen")
       time.sleep(2)                     # Server schonen

2013 13392 Zeilen
2014 13392 Zeilen
2015 13392 Zeilen
2016 13392 Zeilen
2017 13392 Zeilen
2018 13392 Zeilen
2019 13392 Zeilen
2020 13392 Zeilen
2021 13392 Zeilen
2022 13392 Zeilen
2023 13392 Zeilen
2024 13392 Zeilen
2025 13392 Zeilen
2026 13392 Zeilen


In [8]:
   rows = []
   for f in sorted(RAW.glob("hesta_201_*.json")):
       for d in json.loads(f.read_text(encoding="utf-8"))["data"]:
           rows.append(d["key"] + d["values"])          # [Jahr, Monat, Gemeinde, Indikator, Wert]
   df = pd.DataFrame(rows, columns=["jahr", "monat", "bfs_nr", "indikator", "wert"])
   df["wert"] = pd.to_numeric(df["wert"], errors="coerce")   # "..." -> NaN
   namen = {"2":"zimmer","3":"betten","5":"logiernaechte","6":"zimmernaechte",
            "7":"zimmerauslastung","8":"bettenauslastung"}
   df["indikator"] = df["indikator"].map(namen)
   hotel = df.pivot_table(index=["bfs_nr","jahr","monat"], columns="indikator",
                          values="wert", aggfunc="first").reset_index()
   hotel[["bfs_nr","jahr","monat"]] = hotel[["bfs_nr","jahr","monat"]].astype(int)
   hotel.to_csv("../data/processed/hotellerie_monat.csv", index=False)
   print(hotel.shape); hotel.head()

(29459, 9)


indikator,bfs_nr,jahr,monat,betten,bettenauslastung,logiernaechte,zimmer,zimmerauslastung,zimmernaechte
0,1024,2013,1,310.0,18.0,1737.0,175.0,22.0,1181.0
1,1024,2013,10,310.0,29.0,2760.0,175.0,44.0,2364.0
2,1024,2013,11,310.0,24.0,2186.0,175.0,35.0,1830.0
3,1024,2013,12,310.0,20.0,1882.0,175.0,30.0,1575.0
4,1024,2013,2,310.0,19.0,1684.0,175.0,31.0,1504.0


In [9]:
   print(hotel.shape)                         # max. 186 x 12 x 14 = 31'248 Zeilen
   print(hotel.isna().sum())                  # fehlende Werte pro Spalte, Zahl notieren
   print(hotel.describe().round(1))           # zimmerauslastung muss zwischen 0 und 100 liegen
   print(sorted(hotel["jahr"].unique()))      # 2013 bis 2026

(29459, 9)
indikator
bfs_nr               0
jahr                 0
monat                0
betten               0
bettenauslastung    81
logiernaechte       81
zimmer               0
zimmerauslastung    81
zimmernaechte       81
dtype: int64
indikator   bfs_nr     jahr    monat   betten  bettenauslastung  \
count      29459.0  29459.0  29459.0  29459.0           29378.0   
mean        3244.7   2019.4      6.4   1007.2              36.9   
std         2076.2      3.9      3.4   1832.3              17.7   
min           62.0   2013.0      1.0      4.0               0.0   
25%         1202.0   2016.0      3.0    235.0              24.0   
50%         3298.0   2019.0      6.0    445.0              35.0   
75%         5236.0   2023.0      9.0    859.0              48.0   
max         6831.0   2026.0     12.0  19553.0             168.0   

indikator  logiernaechte   zimmer  zimmerauslastung  zimmernaechte  
count            29378.0  29459.0           29378.0        29378.0  
mean             

## 3. Gäste-Herkunft (Tabelle 101)

In [10]:
meta101 = json.loads((RAW / "meta_101.json").read_text(encoding="utf-8"))
gemeinden101 = next(v for v in meta101["variables"] if v["code"] == "Gemeinde")["values"]

def fetch_101(jahr):
    query = {
        "query": [
            {"code": "Jahr",          "selection": {"filter": "item", "values": [str(jahr)]}},
            {"code": "Monat",         "selection": {"filter": "item", "values": [str(m) for m in range(1, 13)]}},
            {"code": "Gemeinde",      "selection": {"filter": "item", "values": gemeinden101}},
            {"code": "Herkunftsland", "selection": {"filter": "item", "values": ["0", "1"]}},
            {"code": "Indikator",     "selection": {"filter": "item", "values": ["2"]}},
        ],
        "response": {"format": "json"},
    }
    r = requests.post(BASE.format(t="101"), json=query, timeout=60)
    r.raise_for_status()
    return r.json()

for jahr in range(2013, 2027):
    ziel = RAW / f"hesta_101_{jahr}.json"
    if ziel.exists():
        continue
    data = fetch_101(jahr)
    ziel.write_text(json.dumps(data), encoding="utf-8")
    print(jahr, len(data["data"]), "Zeilen")
    time.sleep(2)

2013 4464 Zeilen
2014 4464 Zeilen
2015 4464 Zeilen
2016 4464 Zeilen
2017 4464 Zeilen
2018 4464 Zeilen
2019 4464 Zeilen
2020 4464 Zeilen
2021 4464 Zeilen
2022 4464 Zeilen
2023 4464 Zeilen
2024 4464 Zeilen
2025 4464 Zeilen
2026 4464 Zeilen


In [11]:
rows = []
for f in sorted(RAW.glob("hesta_101_*.json")):
    for d in json.loads(f.read_text(encoding="utf-8"))["data"]:
        rows.append(d["key"] + d["values"])     # [Jahr, Monat, Gemeinde, Herkunft, Indikator, Wert]
df101 = pd.DataFrame(rows, columns=["jahr", "monat", "bfs_nr", "herkunft", "indikator", "wert"])
df101["wert"] = pd.to_numeric(df101["wert"], errors="coerce")

herkunft = (df101.pivot_table(index=["bfs_nr", "jahr", "monat"], columns="herkunft",
                              values="wert", aggfunc="first")
            .rename(columns={"0": "ln_total", "1": "ln_schweiz"})
            .reset_index())
herkunft.columns.name = None
herkunft[["bfs_nr", "jahr", "monat"]] = herkunft[["bfs_nr", "jahr", "monat"]].astype(int)

herkunft["ln_ausland"] = herkunft["ln_total"] - herkunft["ln_schweiz"]
herkunft["anteil_ausland"] = (herkunft["ln_ausland"] / herkunft["ln_total"]).where(herkunft["ln_total"] > 0)

herkunft = herkunft[["bfs_nr", "jahr", "monat", "ln_total", "ln_schweiz", "ln_ausland", "anteil_ausland"]]
herkunft.to_csv("../data/processed/herkunft_monat.csv", index=False)
print(herkunft.shape); herkunft.head()

(29378, 7)


,bfs_nr,jahr,monat,ln_total,ln_schweiz,ln_ausland,anteil_ausland
0,1024,2013,1,1737.0,987.0,750.0,0.431779
1,1024,2013,10,2760.0,1381.0,1379.0,0.499638
2,1024,2013,11,2186.0,952.0,1234.0,0.564501
3,1024,2013,12,1882.0,930.0,952.0,0.505845
4,1024,2013,2,1684.0,767.0,917.0,0.544537


In [12]:
print(herkunft.shape)
print(herkunft.isna().sum())
print(herkunft.describe().round(2))
print("negatives ln_ausland:", (herkunft["ln_ausland"] < 0).sum())

chk = hotel.merge(herkunft, on=["bfs_nr", "jahr", "monat"], how="inner")
diff = (chk["logiernaechte"] - chk["ln_total"]).abs()
print("verglichene Zeilen:", len(chk), "| Abweichungen > 0:", (diff > 0).sum())

(29378, 7)
bfs_nr            0
jahr              0
monat             0
ln_total          0
ln_schweiz        0
ln_ausland        0
anteil_ausland    0
dtype: int64
         bfs_nr      jahr     monat   ln_total  ln_schweiz  ln_ausland  \
count  29378.00  29378.00  29378.00   29378.00    29378.00    29378.00   
mean    3244.10   2019.43      6.40   13574.23     6293.98     7280.25   
std     2075.75      3.95      3.43   30848.03    10580.96    22145.34   
min       62.00   2013.00      1.00       1.00        0.00        0.00   
25%     1202.00   2016.00      3.00    1837.00     1162.00      477.00   
50%     3298.00   2019.00      6.00    4145.00     2464.00     1388.00   
75%     5210.00   2023.00      9.00   10989.25     6436.00     4389.50   
max     6831.00   2026.00     12.00  424162.00   124342.00   305266.00   

       anteil_ausland  
count        29378.00  
mean             0.38  
std              0.20  
min              0.00  
25%              0.22  
50%              0.38  
7

## 4. Gemeinde-Stammdaten (Koordinaten, Höhe)

In [13]:
   import requests, time, json, pathlib
   import pandas as pd

   meta = json.loads((RAW / "meta_201.json").read_text(encoding="utf-8"))
   g = next(v for v in meta["variables"] if v["code"] == "Gemeinde")
   gem = pd.DataFrame({"bfs_nr": g["values"], "name": g["valueTexts"]})
   gem["bfs_nr"] = gem["bfs_nr"].astype(int)
   gem["suchname"] = gem["name"].str.replace(r"\s*\(.*\)", "", regex=True)     # "Brienz (BE)" -> "Brienz"
   gem["kt_hint"] = gem["name"].str.extract(r"\((\w\w)\)")                       # "BE"
   print(gem["kt_hint"].notna().sum(), "Gemeinden mit Kantonshinweis")
   gem.head(10)

8 Gemeinden mit Kantonshinweis


,bfs_nr,name,suchname,kt_hint
0,62,Kloten,Kloten,NaN
1,66,Opfikon,Opfikon,NaN
2,191,Dübendorf,Dübendorf,NaN
3,198,Uster,Uster,NaN
4,230,Winterthur,Winterthur,NaN
5,261,Zürich,Zürich,NaN
6,293,Wädenswil,Wädenswil,NaN
7,295,Horgen,Horgen,NaN
8,329,Langenthal,Langenthal,NaN
9,351,Bern,Bern,NaN


In [14]:
   KT = {"ZH": ["zürich", "zurich"], "BE": ["bern"], "LU": ["luzern", "lucerne"], "UR": ["uri"],
         "SZ": ["schwyz"], "OW": ["obwalden"], "NW": ["nidwalden"], "GL": ["glarus"], "ZG": ["zug"],
         "FR": ["freiburg", "fribourg"], "SO": ["solothurn"], "BS": ["basel-stadt"],
         "BL": ["basel-landschaft"], "SH": ["schaffhausen"], "AR": ["ausserrhoden"],
         "AI": ["innerrhoden"], "SG": ["st. gallen", "saint gall"], "GR": ["graubünden", "grisons", "grigioni"],
         "AG": ["aargau"], "TG": ["thurgau"], "TI": ["tessin", "ticino"], "VD": ["waadt", "vaud"],
         "VS": ["wallis", "valais"], "NE": ["neuenburg", "neuchâtel", "neuchatel"],
         "GE": ["genf", "genève", "geneva"], "JU": ["jura"]}

   def geocode(name):
       r = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                        params={"name": name, "count": 10, "language": "de",
                                "countryCode": "CH", "format": "json"}, timeout=30)
       r.raise_for_status()
       return r.json().get("results", [])

   def waehle(res, kt):
       if not res:
           return None, False
       if isinstance(kt, str) and kt in KT:
           for r in res:
               if any(k in r.get("admin1", "").lower() for k in KT[kt]):
                   return r, True
           return res[0], False            # kein Treffer im richtigen Kanton
       return res[0], True

   treffer = []
   for row in gem.itertuples():
       res = geocode(row.suchname)
       best, kt_ok = waehle(res, row.kt_hint)
       best = best or {}
       treffer.append({"bfs_nr": row.bfs_nr, "lat": best.get("latitude"), "lon": best.get("longitude"),
                       "hoehe_m": best.get("elevation"), "ort_gefunden": best.get("name"),
                       "kanton_geo": best.get("admin1"), "n_treffer": len(res), "kt_ok": kt_ok})
       time.sleep(0.5)

   gem = gem.merge(pd.DataFrame(treffer), on="bfs_nr")
   gem.to_csv("../data/processed/gemeinden_roh.csv", index=False)
   print("ohne Treffer:", gem["lat"].isna().sum())
   print("Kanton passt nicht:", (~gem["kt_ok"] & gem["lat"].notna()).sum())
   print("mehrere Treffer:", (gem["n_treffer"] > 1).sum())

ohne Treffer: 23
Kanton passt nicht: 1
mehrere Treffer: 66


In [ ]:
   print(gem.sort_values("hoehe_m")[["bfs_nr", "name", "ort_gefunden", "kanton_geo", "hoehe_m", "n_treffer", "kt_ok"]]
         .to_string(index=False))

## 4. Gemeinde-Stammdaten (Koordinaten, Höhe)

In [2]:
import requests, time, json, pathlib
import pandas as pd

RAW = pathlib.Path("../data/raw/bfs")

meta = json.loads((RAW / "meta_201.json").read_text(encoding="utf-8"))
g = next(v for v in meta["variables"] if v["code"] == "Gemeinde")
gem = pd.DataFrame({"bfs_nr": g["values"], "name": g["valueTexts"]})
gem["bfs_nr"] = gem["bfs_nr"].astype(int)
gem["suchname"] = gem["name"].str.replace(r"\s*\(.*\)", "", regex=True)     # "Brienz (BE)" -> "Brienz"
gem["kt_hint"] = gem["name"].str.extract(r"\((\w\w)\)")                       # "BE"
print(gem["kt_hint"].notna().sum(), "Gemeinden mit Kantonshinweis")
gem.head(10)

8 Gemeinden mit Kantonshinweis


,bfs_nr,name,suchname,kt_hint
0,62,Kloten,Kloten,NaN
1,66,Opfikon,Opfikon,NaN
2,191,Dübendorf,Dübendorf,NaN
3,198,Uster,Uster,NaN
4,230,Winterthur,Winterthur,NaN
5,261,Zürich,Zürich,NaN
6,293,Wädenswil,Wädenswil,NaN
7,295,Horgen,Horgen,NaN
8,329,Langenthal,Langenthal,NaN
9,351,Bern,Bern,NaN


In [3]:
   KT = {"ZH": ["zürich", "zurich"], "BE": ["bern"], "LU": ["luzern", "lucerne"], "UR": ["uri"],
         "SZ": ["schwyz"], "OW": ["obwalden"], "NW": ["nidwalden"], "GL": ["glarus"], "ZG": ["zug"],
         "FR": ["freiburg", "fribourg"], "SO": ["solothurn"], "BS": ["basel-stadt"],
         "BL": ["basel-landschaft"], "SH": ["schaffhausen"], "AR": ["ausserrhoden"],
         "AI": ["innerrhoden"], "SG": ["st. gallen", "saint gall"], "GR": ["graubünden", "grisons", "grigioni"],
         "AG": ["aargau"], "TG": ["thurgau"], "TI": ["tessin", "ticino"], "VD": ["waadt", "vaud"],
         "VS": ["wallis", "valais"], "NE": ["neuenburg", "neuchâtel", "neuchatel"],
         "GE": ["genf", "genève", "geneva"], "JU": ["jura"]}

   def geocode(name):
       r = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                        params={"name": name, "count": 10, "language": "de",
                                "countryCode": "CH", "format": "json"}, timeout=30)
       r.raise_for_status()
       return r.json().get("results", [])

   def waehle(res, kt):
       if not res:
           return None, False
       if isinstance(kt, str) and kt in KT:
           for r in res:
               if any(k in r.get("admin1", "").lower() for k in KT[kt]):
                   return r, True
           return res[0], False            # kein Treffer im richtigen Kanton
       return res[0], True

   treffer = []
   for row in gem.itertuples():
       res = geocode(row.suchname)
       best, kt_ok = waehle(res, row.kt_hint)
       best = best or {}
       treffer.append({"bfs_nr": row.bfs_nr, "lat": best.get("latitude"), "lon": best.get("longitude"),
                       "hoehe_m": best.get("elevation"), "ort_gefunden": best.get("name"),
                       "kanton_geo": best.get("admin1"), "n_treffer": len(res), "kt_ok": kt_ok})
       time.sleep(0.5)

   gem = gem.merge(pd.DataFrame(treffer), on="bfs_nr")
   gem.to_csv("../data/processed/gemeinden_roh.csv", index=False)
   print("ohne Treffer:", gem["lat"].isna().sum())
   print("Kanton passt nicht:", (~gem["kt_ok"] & gem["lat"].notna()).sum())
   print("mehrere Treffer:", (gem["n_treffer"] > 1).sum())

ohne Treffer: 23
Kanton passt nicht: 1
mehrere Treffer: 66


In [4]:
   print(gem.sort_values("hoehe_m")[["bfs_nr", "name", "ort_gefunden", "kanton_geo", "hoehe_m", "n_treffer", "kt_ok"]]
         .to_string(index=False))

 bfs_nr                     name             ort_gefunden                    kanton_geo  hoehe_m  n_treffer  kt_ok
   5091                   Ascona                   Ascona                 Kanton Tessin    202.0          1   True
   5113                  Locarno                  Locarno                 Kanton Tessin    206.0          2   True
   5120                  Muralto                  Muralto                 Kanton Tessin    221.0          1   True
   5002               Bellinzona               Bellinzona                 Kanton Tessin    230.0          1   True
   5250                  Chiasso                  Chiasso                 Kanton Tessin    237.0          1   True
   4258              Rheinfelden      Rheinfelden, Aargau                 Kanton Aargau    277.0          1   True
   2701                    Basel                    Basel            Kanton Basel-Stadt    279.0          2   True
   5192                   Lugano                   Lugano                 Kanton

In [6]:
print("--- OHNE TREFFER ---")
print("\n".join(f"{r.bfs_nr}|{r.name}" for r in gem[gem["lat"].isna()].itertuples()))
print("--- ALLE (nach Höhe) ---")
print("\n".join(f"{r.bfs_nr}|{r.name}|{r.ort_gefunden}|{r.kanton_geo}|{r.hoehe_m}|{r.n_treffer}|{int(r.kt_ok)}"
                for r in gem.dropna(subset=["lat"]).sort_values("hoehe_m").itertuples()))

--- OHNE TREFFER ---
1630|Glarus Nord
1631|Glarus Süd
2121|Haut-Intyamon
3340|Rapperswil-Jona
3359|Wildhaus-Alt St. Johann
3396|Neckertal
3543|Surses
3618|Lumnezia
3619|Ilanz/Glion
3792|Bregaglia
3847|Val Müstair
3986|Tujetsch
5236|Collina d'Oro
5238|Monteceneri
5411|Ormont-Dessus
5613|Bourg-en-Lavaux
5892|Blonay - Saint-Légier
6002|Brig-Glis
6037|Val de Bagnes
6077|Goms
6252|Anniviers
6417|La Grande Béroche
6513|Laténa
--- ALLE (nach Höhe) ---
5091|Ascona|Ascona|Kanton Tessin|202.0|1|1
5113|Locarno|Locarno|Kanton Tessin|206.0|2|1
5120|Muralto|Muralto|Kanton Tessin|221.0|1|1
5002|Bellinzona|Bellinzona|Kanton Tessin|230.0|1|1
5250|Chiasso|Chiasso|Kanton Tessin|237.0|1|1
4258|Rheinfelden|Rheinfelden, Aargau|Kanton Aargau|277.0|1|1
2701|Basel|Basel|Kanton Basel-Stadt|279.0|2|1
5192|Lugano|Lugano|Kanton Tessin|284.0|3|1
2831|Pratteln|Pratteln|Kanton Basel-Landschaft|291.0|1|1
5210|Paradiso|Paradiso TI|Kanton Tessin|296.0|1|1
2829|Liestal|Liestal|Kanton Basel-Landschaft|327.0|1|1
5254|Mendr

### Korrektur der Gemeinden ohne oder mit falschem Treffer

In [11]:
korr = pd.read_csv("../data/manual/gemeinde_korrektur.csv", encoding="utf-8")
korr_rows = []
for row in korr.itertuples():
    res = geocode(row.referenzort)
    best, kt_ok = waehle(res, row.kt)
    best = best or {}
    korr_rows.append({"bfs_nr": row.bfs_nr, "lat_k": best.get("latitude"), "lon_k": best.get("longitude"),
                      "hoehe_k": best.get("elevation"), "ort_k": best.get("name"), "kt_ok_k": kt_ok})
    time.sleep(0.5)
korr_geo = pd.DataFrame(korr_rows)
print(korr.merge(korr_geo, on="bfs_nr")[["bfs_nr", "name", "referenzort", "ort_k", "hoehe_k", "kt_ok_k"]].to_string(index=False))

 bfs_nr                    name       referenzort                  ort_k  hoehe_k  kt_ok_k
   1630             Glarus Nord            Näfels                 Näfels    437.0     True
   1631              Glarus Süd               Elm                 Elm GL    977.0     True
   2121           Haut-Intyamon           Albeuve                Albeuve    759.0     True
   3340         Rapperswil-Jona        Rapperswil          Rapperswil SG    408.0     True
   3359 Wildhaus-Alt St. Johann          Wildhaus            Wildhaus SG   1090.0     True
   3396               Neckertal        Brunnadern             Brunnadern    659.0     True
   3543                  Surses          Savognin               Savognin   1219.0     True
   3618                Lumnezia             Villa                  Villa   1253.0     True
   3619             Ilanz/Glion             Ilanz                  Ilanz    705.0     True
   3792               Bregaglia            Soglio              Soglio GR   1099.0     True

### Korrekturen übernehmen


In [12]:
gem_k = gem.merge(korr_geo, on="bfs_nr", how="left")
m = gem_k["lat_k"].notna()
for a, b in [("lat", "lat_k"), ("lon", "lon_k"), ("hoehe_m", "hoehe_k"), ("ort_gefunden", "ort_k")]:
    gem_k.loc[m, a] = gem_k.loc[m, b]
gem_k["korrigiert"] = m
gem_k = gem_k.drop(columns=["lat_k", "lon_k", "hoehe_k", "ort_k", "kt_ok_k"])
print("korrigiert:", gem_k["korrigiert"].sum(), "| noch ohne Koordinaten:", gem_k["lat"].isna().sum())

korrigiert: 34 | noch ohne Koordinaten: 0


### Höhenklassen bilden und Gemeinde-Stammdaten speichern


In [13]:
   def klasse(h):
       return "tief" if h < 1000 else ("mittel" if h <= 1500 else "hoch")

   gem_k["hoehenklasse"] = gem_k["hoehe_m"].apply(klasse)
   print(gem_k["hoehenklasse"].value_counts())

   gem_k["kanton"] = gem_k["kanton_geo"].fillna(gem_k["kt_hint"])
   out = gem_k[["bfs_nr", "name", "kanton", "lat", "lon", "hoehe_m", "hoehenklasse", "ort_gefunden", "korrigiert"]]
   out.to_csv("../data/processed/gemeinden.csv", index=False)

hoehenklasse
tief      134
mittel     39
hoch       13
Name: count, dtype: int64


### Restliche Gemeinden anhand API suchen: Test

In [15]:
liste = gem_k.sort_values("hoehe_m")[["bfs_nr", "name", "ort_gefunden", "kanton_geo", "hoehe_m", "n_treffer", "kt_ok", "korrigiert"]]
liste.to_csv("../data/processed/gemeinden_liste.txt", sep="|", index=False)
print(len(liste))

186
